# 10 End-to-End Demo

This notebook ties together the complete local workflow and shows dynamic adapter switching.

```mermaid
flowchart LR
    A[Generate training data] --> B[Train LoRA adapters]
    B --> C[Register in MLflow]
    C --> D[Start vLLM]
    D --> E[Load adapters]
    E --> F[FastAPI gateway]
    F --> G[Domain-specific responses]
```

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Local workflow checklist

Run notebooks `01` through `09` in order for the full live demo. The cells below provide a compact command checklist.

In [ ]:
commands = [
    "make up",
    "make notebooks",
    "python training/generate_synthetic.py",
    "python training/train_lora.py --adapter finance",
    "python training/train_lora.py --adapter legal",
    "python training/train_lora.py --adapter healthcare",
    "python training/register_mlflow.py",
    "make serve",
    "python scripts/load_adapters.py",
    "make api",
    "python scripts/test_inference.py --adapter finance",
]
print("\n".join(commands))

## Dynamic adapter switching through the gateway

Expected output: each request returns `routed_adapter` matching the domain.

In [ ]:
import requests

demo_prompts = [
    "Explain a revenue forecast risk.",
    "Explain a contract indemnity clause.",
    "Explain patient discharge instructions.",
]

for prompt in demo_prompts:
    payload = {"messages": [{"role": "user", "content": prompt}], "max_tokens": 140}
    response = requests.post(f"{settings_cfg.api_base_url}/chat", json=payload, timeout=60)
    print("\nPROMPT:", prompt)
    print(response.status_code)
    print(response.text[:1000])

## MLIS transfer summary

Copy `adapters/`, `.env`, `scripts/load_adapters.py`, and the serving configuration to the MLIS host. Set `VLLM_BASE_URL` to the remote vLLM endpoint and run the same adapter loading script. The trained adapters remain standalone PEFT artifacts.